<a href="https://colab.research.google.com/github/husthorng/Backpropagation_NN/blob/main/ANN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import auth
import gspread
from google.auth import default
auth.authenticate_user()

import numpy as np

def logistic(x):
    return 1 / (1 + np.exp(-x))

maxcycles = 90000
SSE_Goal = 0.1
lr = 0.001

SSE_log = []


creds, _ = default()
gc = gspread.authorize(creds)
#在雲端硬碟 建立速算表 cloud_data 工作表 data
#https://docs.google.com/spreadsheets/d/15kZWq3YmtF5yx40BRuz6TMGx0D22_r9QO-ea84BaRKQ/edit?gid=0#gid=0
sh = gc.open("cloud_data")
worksheet = sh.worksheet("data")

#data = worksheet.get_all_values()
data = worksheet.get_all_values()[1:]

data_np = np.array(data, dtype=float)

min_v = np.min(data_np, axis=0)
max_v = np.max(data_np, axis=0)

data_norm = (data_np - min_v) / (max_v - min_v)
x=data_norm[:,0:4]
t=data_norm[:,4:6]

X = np.hstack((x, np.ones((x.shape[0], 1))))
patterns,inputs  = x.shape
_, outputs = t.shape

hidden = 6

np.random.seed()

W1 = 0.5 * np.random.randn(inputs+1, hidden)
W2 = 0.5 * np.random.randn(hidden+1, outputs)

w=np.load("/content/drive/MyDrive/m/m/W1W2.npz")
W1 = w['W1']
W2 = w['W2']



fileID = open("/content/drive/MyDrive/exp.txt","w")

# ===============================
# Training Loop
# ===============================

for k in range(maxcycles):

  # forward
  h = logistic(X @ W1)
  H = np.hstack((h, np.ones((h.shape[0],1))))
  output = logistic(H @ W2)

  e = t - output
  SSE = np.sum(e**2)

  if k % 1000 == 0:
      eti = np.sum(e[0,:]**2)
      ete = np.sum(e[1,:]**2)
      print(f"Epoch={k}, SSE={SSE:.8f}")
      fileID.write(f"{k} {SSE:.8f} {eti:.8f} {ete:.8f}\n")

  if SSE < SSE_Goal:
      print("Goal reached")
      break

  # backprop
  delta2 = output * (1-output) * e
  W2 += 2 * lr * H.T @ delta2

  delta1 = h * (1-h) * (delta2 @ W2[:-1,:].T)
  W1 += 2 * lr * X.T @ delta1

fileID.close()
# Save model
np.savez("/content/drive/MyDrive/m/m/W1W2.npz", W1=W1, W2=W2, max_v=max_v, min_v=min_v)

print("Training Finished")
# ===============================
# Predict
# ===============================
h = logistic(X @ W1)
H = np.hstack((h, np.ones((h.shape[0],1))))
output = logistic(H @ W2)

# 反正規化
OV = output * (max_v - min_v)[4:6] + min_v[4:6]


Epoch=0, SSE=0.12428431
Epoch=1000, SSE=0.12428259
Epoch=2000, SSE=0.12428086
Epoch=3000, SSE=0.12427914
Epoch=4000, SSE=0.12427741
Epoch=5000, SSE=0.12427569
Epoch=6000, SSE=0.12427397
Epoch=7000, SSE=0.12427225
Epoch=8000, SSE=0.12427053
Epoch=9000, SSE=0.12426881
Epoch=10000, SSE=0.12426710
Epoch=11000, SSE=0.12426538
Epoch=12000, SSE=0.12426367
Epoch=13000, SSE=0.12426196
Epoch=14000, SSE=0.12426024
Epoch=15000, SSE=0.12425853
Epoch=16000, SSE=0.12425682
Epoch=17000, SSE=0.12425512
Epoch=18000, SSE=0.12425341
Epoch=19000, SSE=0.12425170
Epoch=20000, SSE=0.12425000
Epoch=21000, SSE=0.12424829
Epoch=22000, SSE=0.12424659
Epoch=23000, SSE=0.12424489
Epoch=24000, SSE=0.12424319
Epoch=25000, SSE=0.12424149
Epoch=26000, SSE=0.12423979
Epoch=27000, SSE=0.12423809
Epoch=28000, SSE=0.12423640
Epoch=29000, SSE=0.12423470
Epoch=30000, SSE=0.12423301
Epoch=31000, SSE=0.12423132
Epoch=32000, SSE=0.12422963
Epoch=33000, SSE=0.12422794
Epoch=34000, SSE=0.12422625
Epoch=35000, SSE=0.12422456
Epoch

In [3]:
# ===============================
# Predict
# ===============================
h = logistic(X @ W1)
H = np.hstack((h, np.ones((h.shape[0],1))))
output = logistic(H @ W2)

# 反正規化
OV = output * (max_v - min_v)[4:6] + min_v[4:6]

# ===============================
# Write to Google Sheet
# ===============================

worksheet_out = sh.worksheet("output")

# 轉成 list 才能寫入
OV_list = OV.tolist()

# 可加標題
header = [["Ti_pred","Te_pred"]]

worksheet_out.clear()

worksheet_out.update("A1", header + OV_list)

print("Output written to Google Sheet")


/tmp/ipykernel_447/1043838957.py:25: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet_out.update("A1", header + OV_list)


Output written to Google Sheet
